환경 설정 & 모델 로드

In [1]:
%cd ..

/home/user/문서/neuromeka_ojh/new code/code/workspace/mcp_servers/detect_server


/home/user/문서/neuromeka_ojh/new code/code/workspace/mcp_servers/detect_server/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:

import os
import cv2
import json
import torch
import numpy as np
import supervision as sv
import pycocotools.mask as mask_util
from pathlib import Path
from supervision.draw.color import ColorPalette
from PIL import Image
import sys

print("Current working dir:", os.getcwd())

GROUND_SAM2_PATH = os.path.join(os.getcwd(), "Grounded-SAM-2")

sys.path.append(GROUND_SAM2_PATH)
# transformers 기반 (huggingface) GroundingDINO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# 사용자 정의 color map (예시)
CUSTOM_COLORS = [
    "#4365F0",  # blue
    "#F04343",  # red
    "#43F059",  # green
]

# 1) 디바이스 설정 (GPU 가능 시 GPU, 아니면 CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 2) bfloat16 / TF32 최적화 (Ampere 이후 GPU)
torch.autocast(device_type=DEVICE, dtype=torch.bfloat16).__enter__()
if DEVICE == "cuda":
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

# 3) SAM2 모델 설정 (가장 가벼운 Tiny)
SAM2_CHECKPOINT = os.path.join(GROUND_SAM2_PATH, "checkpoints", "sam2.1_hiera_tiny.pt")

SAM2_MODEL_CONFIG = "configs/sam2.1/sam2.1_hiera_t.yaml"

# 실제로는 build_sam2, SAM2ImagePredictor import 후 사용
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
dino_model_id="IDEA-Research/grounding-dino-tiny"
sam2_config_path="configs/sam2.1/sam2.1_hiera_t.yaml" # 경로 오류 뜨면 Grounded-SAM-2 폴더 내부 위치 넣기
sam2_ckpt_path = "Grounded-SAM-2/checkpoints/sam2.1_hiera_tiny.pt" # 경로 오류 뜨면 Grounded-SAM-2 폴더 내부 위치 넣기
print("Loading SAM2 model...")
# sam2_model = build_sam2(SAM2_MODEL_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
# sam2_predictor = SAM2ImagePredictor(sam2_model)
sam2_model = build_sam2(sam2_config_path, sam2_ckpt_path, device="cuda")
sam2_predictor = SAM2ImagePredictor(sam2_model)
print("SAM2 loaded.")

# 4) Grounding DINO (Tiny)
GROUNDING_MODEL = "IDEA-Research/grounding-dino-tiny"
processor = AutoProcessor.from_pretrained(GROUNDING_MODEL)
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_MODEL).to(DEVICE)
print("Grounding DINO (Tiny) loaded on", DEVICE)

print("\n✅ All models are loaded and ready!")

Current working dir: /home/user/문서/neuromeka_ojh/new code/code/workspace/mcp_servers/detect_server


Loading SAM2 model...
SAM2 loaded.
Grounding DINO (Tiny) loaded on cuda

✅ All models are loaded and ready!


추론 함수 정의

In [3]:
import pyrealsense2 as rs
import numpy as np
import cv2
import time

# --- RealSense 카메라 제어 클래스 (이전 답변의 개선된 버전) ---
class RealSenseCapture:
    def __init__(self, depth_w=848, depth_h=480, color_w=848, color_h=480, fps=30):
        """카메라 설정 초기화"""
        self.pipeline = rs.pipeline()
        self.config = rs.config()
        self.align = None
        self.is_running = False
        self.depth_scale = None
        self.depth_intrinsics = None
        self.color_intrinsics = None # 컬러 기준으로 정렬 및 언프로젝션 시 사용

        self.depth_w, self.depth_h = depth_w, depth_h
        self.color_w, self.color_h = color_w, color_h
        self.fps = fps

        print("스트림 설정 시도...")
        try:
            self.config.enable_stream(rs.stream.depth, self.depth_w, self.depth_h, rs.format.z16, self.fps)
            self.config.enable_stream(rs.stream.color, self.color_w, self.color_h, rs.format.bgr8, self.fps)

            print("스트림 설정 완료.")
        except RuntimeError as e:
            print(f"지원하지 않는 스트림 설정입니다: {e}")
            self.pipeline = None # 파이프라인 사용 불가 표시
            raise

    def start(self):
        """파이프라인 시작 및 정보 가져오기"""
        if not self.pipeline: return False # 초기화 실패 시 시작 불가
        if not self.is_running:
            print("파이프라인 시작 중...")
            try:
                profile = self.pipeline.start(self.config)
                self.is_running = True
                # # --- 자동 화이트밸런스/노출 비활성화 및 수동 설정 ---
                # color_sensor = profile.get_device().first_color_sensor()
                # color_sensor.set_option(rs.option.enable_auto_white_balance, 0)
                # color_sensor.set_option(rs.option.enable_auto_exposure,     0)
                # color_sensor.set_option(rs.option.white_balance,             4800)   # 4800K
                # color_sensor.set_option(rs.option.exposure,                  33000)  # 33ms
                # color_sensor.set_option(rs.option.gain,                      16)
                # # ----------------------------------------------
                print("파이프라인 시작 완료.")

                # 깊이 스케일 가져오기
                depth_sensor = profile.get_device().first_depth_sensor()
                self.depth_scale = depth_sensor.get_depth_scale()
                print(f"Depth Scale: {self.depth_scale}")

                # 정렬 객체 생성 (컬러 스트림 기준)
                align_to = rs.stream.color
                self.align = rs.align(align_to)

                # 내부 파라미터(Intrinsics) 가져오기 (정렬 기준인 컬러 스트림 사용)
                color_stream = profile.get_stream(rs.stream.color).as_video_stream_profile()
                self.color_intrinsics = color_stream.get_intrinsics()
                print(f"Color Intrinsics: fx={self.color_intrinsics.fx}, fy={self.color_intrinsics.fy}, "
                      f"cx={self.color_intrinsics.ppx}, cy={self.color_intrinsics.ppy}")

                # 초기 안정화
                print("초기 안정화 대기 중...")
                for _ in range(60):
                    self.pipeline.wait_for_frames()
                print("초기 안정화 완료.")
                return True

            except RuntimeError as e:
                print(f"파이프라인 시작 실패: {e}")
                self.is_running = False
                return False
        else:
            print("파이프라인이 이미 실행 중입니다.")
            return True

    def capture_aligned_frames(self, timeout_ms=2000):
        """정렬된 컬러 및 깊이 프레임 캡처 (NumPy 배열 반환)"""

        spatial = rs.spatial_filter()
        temporal = rs.temporal_filter()

        # 필터 옵션 설정 (선택 사항, 필요에 따라 값 조정)
        spatial.set_option(rs.option.filter_magnitude, 2) # 필터 강도 (1-5)
        spatial.set_option(rs.option.filter_smooth_alpha, 0.5) # 스무딩 강도 (0.25-1)
        spatial.set_option(rs.option.filter_smooth_delta, 20) # 엣지 보존 강도 (1-50)
        # spatial.set_option(rs.option.holes_fill, 0) # 홀 채우기 (0: 없음, 1: 인접 픽셀, 2: 가장 먼 곳 ...)

        temporal.set_option(rs.option.filter_smooth_alpha, 0.4) # 과거 프레임 가중치 (0-1)
        temporal.set_option(rs.option.filter_smooth_delta, 1) # 변화량 임계값 (1-100)
        if not self.is_running:
            print("오류: 파이프라인 미실행.")
            return None, None

        try:
            frames = self.pipeline.wait_for_frames(timeout_ms)
            if not frames:
                print("오류: 프레임 수신 실패 (타임아웃).")
                return None, None

            aligned_frames = self.align.process(frames)
            depth_frame = aligned_frames.get_depth_frame()
            color_frame = aligned_frames.get_color_frame()
            depth_frame = spatial.process(depth_frame)
            depth_frame = temporal.process(depth_frame)

            if not depth_frame or not color_frame:
                print("오류: 유효한 프레임 획득 실패.")
                return None, None

            # 중요: 깊이 프레임은 원본(uint16) 그대로 반환, 스케일은 별도 관리
            depth_image_raw = np.asanyarray(depth_frame.get_data())
            color_image = np.asanyarray(color_frame.get_data())

            return color_image, depth_image_raw # 깊이 원본 반환

        except Exception as e:
            print(f"프레임 캡처 중 오류: {e}")
            return None, None

    def get_intrinsics(self):
        """컬러 카메라 내부 파라미터 반환"""
        return self.color_intrinsics

    def get_depth_scale(self):
        """깊이 스케일 반환"""
        return self.depth_scale

    def stop(self):
        """파이프라인 중지"""
        if self.is_running:
            print("파이프라인 중지...")
            self.pipeline.stop()
            self.is_running = False
            print("파이프라인 중지 완료.")

    def __enter__(self):
        if not self.start():
             raise RuntimeError("RealSense 카메라 시작 실패")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.stop()

In [4]:
capture= RealSenseCapture()
capture.start()

스트림 설정 시도...
스트림 설정 완료.
파이프라인 시작 중...
파이프라인 시작 완료.
Depth Scale: 0.0010000000474974513
Color Intrinsics: fx=605.5885620117188, fy=605.7113647460938, cx=430.3742980957031, cy=249.6859588623047
초기 안정화 대기 중...
초기 안정화 완료.


True

여러 이미지/프롬프트 테스트

In [5]:
from sklearn.decomposition import PCA

def calculate_object_properties(mask, depth_image_raw, intrinsics, depth_scale): # added in 7_31 but PCA remains
    try:
        # 1) 입력 검증 
        if intrinsics is None or depth_scale is None:
            print("Error: Invalid camera intrinsics or depth scale.")
            return None, None, None, None
        if mask.shape != depth_image_raw.shape:
            print("Error: Mask shape mismatch.")
            return None, None, None, None
        if mask.dtype != bool:
            mask = mask > 0
        
        pts_yx = np.argwhere(mask)
        if pts_yx.size == 0:
            print("Warning: No mask pixels.")
            return None, None, None, None
        
        # 2) 깊이 → 미터 (한 번에 처리)
        depth_m = depth_image_raw.astype(np.float32) * depth_scale
        
        # 3) 벡터화된 3D 변환
        # 유효한 깊이 값만 필터링
        y_coords, x_coords = pts_yx[:, 0], pts_yx[:, 1]
        depth_vals = depth_m[y_coords, x_coords]
        valid_mask = (depth_vals > 0.01) & (depth_vals < 0.8)
        
        if np.sum(valid_mask) < 4:
            print("Warning: Too few 3D points.")
            return None, None, None, None
        
        # 유효한 점들만 선택
        valid_x = x_coords[valid_mask]
        valid_y = y_coords[valid_mask]
        valid_depths = depth_vals[valid_mask]
        
        # 벡터화된 deproject 계산 
        fx, fy = intrinsics.fx, intrinsics.fy
        ppx, ppy = intrinsics.ppx, intrinsics.ppy
        
        # 3D 좌표 계산 (벡터화)
        X = (valid_x - ppx) * valid_depths / fx
        Y = (valid_y - ppy) * valid_depths / fy
        Z = valid_depths
        
        pc = np.column_stack((X, Y, Z)).astype(np.float32)
        
        # 4) 중심점 계산
        center_3d = pc.mean(axis=0)
        
        # 5) 절대 높이
        height = float(pc[:, 2].max() - pc[:, 2].min())
        
        # 6) PCA → OBB 축 길이 
        pca = PCA(n_components=3, svd_solver='randomized', random_state=42)
        pca.fit(pc)
        
        t_pc = pca.transform(pc)
        size_obb = t_pc.max(axis=0) - t_pc.min(axis=0)
        
        # 7) 높이에 가장 가까운 축 제거 → L, W
        diffs = np.abs(size_obb - height)
        idx_h = np.argmin(diffs)
        
        # 마스킹을 이용한 효율적인 L, W 계산
        mask_lw = np.ones(3, dtype=bool)
        mask_lw[idx_h] = False
        lw = size_obb[mask_lw]
        L, W = float(lw.max()), float(lw.min())
        obb_dims = [L, W, height]
        
        # 8) tilt/rot: 수평면상 가장 긴 축 기준
        axes = pca.components_
        proj = np.hypot(axes[:, 0], axes[:, 1])
        idx_horiz = np.argmax(proj)
        v_h = axes[idx_horiz]
        
        norm_h = np.linalg.norm(v_h)
        tilt_deg = float(np.degrees(np.arcsin(np.abs(v_h[2]) / norm_h)))
        rot_deg = float(np.degrees(np.arctan2(-v_h[1], v_h[0])))
        
        # 각도 정규화
        if np.abs(rot_deg) > 90:
            rot_deg = (rot_deg + 180) if rot_deg < 0 else (rot_deg - 180)
        


        # 좌표계 변환 (y 반전)
        center_3d = [float(center_3d[0]), float(-center_3d[1]), float(center_3d[2])]
        
        return center_3d, obb_dims, tilt_deg, rot_deg
    
    except Exception as e:
        return

In [ ]:
# === Paper-figures util (paste this in a new cell) ================================================
import os, json, cv2, math, numpy as np
import matplotlib.pyplot as plt

def _ensure_bool_mask(mask):
    return mask.astype(bool) if mask.dtype != bool else mask

def _depth_to_m(depth_image_raw, depth_scale: float):
    d = depth_image_raw.astype(np.float32) * float(depth_scale)
    return d

def _masked_points_xyz_from_intrinsics(mask_bool, depth_m, intrinsics,
                                       z_min=0.01, z_max=0.8):
    """mask+depth -> 카메라 좌표계 XYZ (네 코드의 기준과 동일한 z 필터)"""
    mask_bool = _ensure_bool_mask(mask_bool)
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return np.empty((0,3), np.float32), (np.array([], dtype=np.int32), np.array([], dtype=np.int32))

    z = depth_m[ys, xs].astype(np.float32)
    valid = (z > z_min) & (z < z_max)
    if valid.sum() < 4:
        return np.empty((0,3), np.float32), (np.array([], dtype=np.int32), np.array([], dtype=np.int32))

    xs = xs[valid].astype(np.float32)
    ys = ys[valid].astype(np.float32)
    z  = z[valid]

    fx, fy = intrinsics.fx, intrinsics.fy
    cx, cy = intrinsics.ppx, intrinsics.ppy
    X = (xs - cx) * z / fx
    Y = (ys - cy) * z / fy
    Z = z
    pc = np.stack([X,Y,Z], axis=1).astype(np.float32)
    return pc, (xs.astype(np.int32), ys.astype(np.int32))

def _draw_mask_overlay(image_bgr, mask_bool, center_xy=None, text=None):
    img = image_bgr.copy()
    m = (_ensure_bool_mask(mask_bool).astype(np.uint8))*255
    overlay = img.copy()

    # 반투명 채움
    alpha = 0.45
    overlay[m>0] = (overlay[m>0]*(1-alpha) + np.array([0,255,0])*alpha).astype(np.uint8)

    # 윤곽선
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0,0,255), 2)

    # 중심점
    if center_xy is not None:
        cx, cy = map(int, center_xy)
        cv2.circle(overlay, (cx,cy), 4, (255,0,0), -1)
        if text:
            cv2.putText(overlay, text, (cx+6, cy-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1, cv2.LINE_AA)
    return overlay

def _oriented_bbox(mask_bool):
    m = (_ensure_bool_mask(mask_bool).astype(np.uint8))*255
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None
    cnt = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)           # ((cx,cy),(w,h),angle)
    box  = cv2.boxPoints(rect).astype(np.int32)  # 4x2
    return rect, box

def _depth_colormap_img(depth_m, mask_bool):
    # 마스크 영역 백분위 기반 범위로 normalize
    m = _ensure_bool_mask(mask_bool)
    vals = depth_m[m]
    if vals.size == 0:
        return None
    lo, hi = np.percentile(vals, [2, 98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(vals.min()), float(vals.max())
        if hi <= lo: return None
    norm = np.clip((depth_m - lo) / max(1e-6, (hi-lo)), 0, 1)
    img8 = (norm*255).astype(np.uint8)
    cm = cv2.applyColorMap(img8, cv2.COLORMAP_INFERNO)
    return cm


# === helper: depth grayscale (mask 구간 2~98% 스트레치) =====================
def _depth_grayscale_img(depth_m: np.ndarray, mask_bool: np.ndarray):
    m = _ensure_bool_mask(mask_bool)
    vals = depth_m[m]
    if vals.size == 0:
        return None
    lo, hi = np.percentile(vals, [2, 98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(vals.min()), float(vals.max())
        if hi <= lo:
            return None
    norm = np.clip((depth_m - lo) / max(1e-6, (hi - lo)), 0, 1)
    gray8 = (norm * 255).astype(np.uint8)   # 0~255 단일 채널
    return gray8



# def _save_pointcloud_fig(pc_xyz, out_path, max_points=15000):
#     if pc_xyz.shape[0] == 0:
#         return False
#     sel = np.random.choice(pc_xyz.shape[0], size=min(max_points, pc_xyz.shape[0]), replace=False)
#     P = pc_xyz[sel]
#     from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
#     fig = plt.figure(figsize=(4,4))
#     ax = fig.add_subplot(111, projection='3d')
#     ax.scatter(P[:,0], P[:,1], P[:,2], s=1)
#     ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
#     ax.view_init(elev=18, azim=35)
#     plt.tight_layout()
#     fig.savefig(out_path, dpi=300)
#     plt.close(fig)
#     return True

# === helper: point cloud figure (no grid) ===================================
def _save_pointcloud_fig(pc_xyz, out_path, max_points=15000):
    if pc_xyz.shape[0] == 0:
        return False
    sel = np.random.choice(pc_xyz.shape[0], size=min(max_points, pc_xyz.shape[0]), replace=False)
    P = pc_xyz[sel]
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    fig = plt.figure(figsize=(4,4))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(P[:,0], P[:,1], P[:,2], s=1)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
    ax.view_init(elev=18, azim=35)
    ax.grid(False)  # ← 격자 제거
    plt.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    return True



# === main: save_visuals_for_paper (요구사항 반영 버전) =======================
def save_visuals_for_paper(
    out_dir: str,
    basename: str,
    image_bgr: np.ndarray,
    mask: np.ndarray,
    depth_image_raw: np.ndarray,
    intrinsics,
    depth_scale: float,
    center_3d=None,      # (x,y,z) m
    obb_dims=None,       # [L, W, H] m
    # NEW: RGB+DINO 박스 입력 2가지 중 하나 선택
    image_with_boxes: np.ndarray = None,   # run_inference의 annotated_frame 그대로 넘기기
    boxes_xyxy: np.ndarray = None,         # (N,4) x1,y1,x2,y2 가 있을 때 직접 그리기
    box_labels: list = None                # 라벨 문자열(선택)
):
    """
    - calculate_object_properties(...)를 값 미지정 시 내부 호출.
    - 결과 파일 경로 dict와 metrics 반환.
    - 변경: overlay(반투명 마스크) 제거, depth는 흑백, RGB 타일은 DINO 박스 포함.
    """
    os.makedirs(out_dir, exist_ok=True)
    paths = {}

    # 0) 이진 마스크 저장
    mask_bool = _ensure_bool_mask(mask)
    mask_u8 = (mask_bool.astype(np.uint8))*255
    p_mask = os.path.join(out_dir, f"{basename}_mask.png")
    cv2.imwrite(p_mask, mask_u8); paths["mask"] = p_mask

    # 1) 2D 중심(픽셀)
    cxcy = None
    if mask_u8.any():
        contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            cnt = max(contours, key=cv2.contourArea)
            M = cv2.moments(cnt)
            if M["m00"] > 1e-6:
                cxcy = (int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"]))

    # 1-1) RGB + DINO 박스 이미지 준비/저장
    if image_with_boxes is not None:
        rgb_boxes_img = image_with_boxes.copy()
    else:
        rgb_boxes_img = image_bgr.copy()
        if boxes_xyxy is not None and len(np.atleast_2d(boxes_xyxy)) > 0:
            boxes_xyxy = np.asarray(boxes_xyxy, dtype=np.float32)
            for i, (x1, y1, x2, y2) in enumerate(boxes_xyxy.astype(int)):
                cv2.rectangle(rgb_boxes_img, (x1, y1), (x2, y2), (0, 255, 255), 2)
                if box_labels is not None and i < len(box_labels):
                    cv2.putText(rgb_boxes_img, str(box_labels[i]), (x1, max(0,y1-5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1, cv2.LINE_AA)
    p_rgb_boxes = os.path.join(out_dir, f"{basename}_rgb_boxes.png")
    cv2.imwrite(p_rgb_boxes, rgb_boxes_img); paths["rgb_boxes"] = p_rgb_boxes

    # 2) 깊이(m) 및 포인트클라우드
    depth_m = _depth_to_m(depth_image_raw, depth_scale)
    pc_xyz, _ = _masked_points_xyz_from_intrinsics(mask_bool, depth_m, intrinsics)
    has_pc = pc_xyz.shape[0] > 0

    # 3) 깊이 "흑백" 맵 & 히스토그램
    gray = _depth_grayscale_img(depth_m, mask_bool)
    if gray is not None:
        p_dgray = os.path.join(out_dir, f"{basename}_depth_gray.png")
        cv2.imwrite(p_dgray, gray); paths["depth_gray"] = p_dgray

    vals = depth_m[mask_bool]
    if vals.size > 0:
        p_hist = os.path.join(out_dir, f"{basename}_depth_hist.png")
        plt.figure(figsize=(4,3))
        plt.hist(vals, bins=64)
        plt.xlabel("Depth (m)"); plt.ylabel("Count"); plt.tight_layout()
        plt.savefig(p_hist, dpi=300)
        plt.close()
        paths["depth_hist"] = p_hist

    # 4) 3D 수치 없으면 계산
    used_calc = False
    if (center_3d is None) or (obb_dims is None):
        try:
            center_3d2, obb_dims2, tilt_deg, rot_deg = calculate_object_properties(
                mask_bool, depth_image_raw, intrinsics, depth_scale
            )
            if center_3d is None: center_3d = center_3d2
            if obb_dims is None:  obb_dims  = obb_dims2
            used_calc = True
        except Exception as e:
            print(f"[warn] calculate_object_properties failed: {e}")

    # 5) 포인트클라우드 그림 (grid 끔)
    if has_pc:
        p_pc = os.path.join(out_dir, f"{basename}_pointcloud.png")
        if _save_pointcloud_fig(pc_xyz, p_pc):
            paths["pointcloud"] = p_pc

    # 6) 2D OBB + L/W 라벨(3D 수치 텍스트만)
    p_dims2d = os.path.join(out_dir, f"{basename}_dims2d.png")
    dims_img = image_bgr.copy()
    rect, box = _oriented_bbox(mask_bool)
    if rect is not None and box is not None:
        cv2.polylines(dims_img, [box], True, (255,0,255), 2)
        p = box
        mid01 = tuple(np.mean([p[0],p[1]], axis=0).astype(int))
        mid12 = tuple(np.mean([p[1],p[2]], axis=0).astype(int))
        L_txt = f"L={obb_dims[0]:.3f} m" if (obb_dims is not None) else "L=N/A"
        W_txt = f"W={obb_dims[1]:.3f} m" if (obb_dims is not None) else "W=N/A"
        cv2.putText(dims_img, L_txt, mid01, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,255), 2, cv2.LINE_AA)
        cv2.putText(dims_img, W_txt, mid12, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,255), 2, cv2.LINE_AA)
    cv2.imwrite(p_dims2d, dims_img); paths["dims2d"] = p_dims2d

    # 7) metrics JSON
    metrics = {
        "center_xyz_m": None if center_3d is None else [float(center_3d[0]), float(center_3d[1]), float(center_3d[2])],
        "L_m": None if (obb_dims is None) else float(obb_dims[0]),
        "W_m": None if (obb_dims is None) else float(obb_dims[1]),
        "H_m": None if (obb_dims is None) else float(obb_dims[2]),
        "used_calculate_object_properties": bool(used_calc),
    }
    p_json = os.path.join(out_dir, f"{basename}_metrics.json")
    with open(p_json, "w") as f:
        json.dump(metrics, f, indent=2)
    paths["metrics_json"] = p_json

    # 8) 패널 합성(6칸) — overlay 대신 구성 변경
    import matplotlib.image as mpimg
    fig, axes = plt.subplots(2,3, figsize=(10,6))
    ax = axes.ravel()

    # (0,0): RGB + DINO boxes
    ax[0].imshow(cv2.cvtColor(rgb_boxes_img, cv2.COLOR_BGR2RGB))
    ax[0].set_title("RGB + DINO boxes"); ax[0].axis("off")

    # (0,1): Binary mask
    ax[1].imshow(mask_bool, cmap="gray")
    ax[1].set_title("Binary mask"); ax[1].axis("off")

    # (0,2): Depth (grayscale)
    if "depth_gray" in paths:
        dimg = mpimg.imread(paths["depth_gray"])
        ax[2].imshow(dimg, cmap="gray"); ax[2].set_title("Depth (grayscale)"); ax[2].axis("off")
    else:
        ax[2].axis("off")

    # (1,0): Depth histogram
    if "depth_hist" in paths:
        himg = mpimg.imread(paths["depth_hist"])
        ax[3].imshow(himg); ax[3].set_title("Depth histogram"); ax[3].axis("off")
    else:
        ax[3].axis("off")

    # (1,1): Masked point cloud
    if "pointcloud" in paths:
        pcimg = mpimg.imread(paths["pointcloud"])
        ax[4].imshow(pcimg); ax[4].set_title("Masked point cloud"); ax[4].axis("off")
    else:
        ax[4].axis("off")

    # (1,2): 2D dims
    ax[5].imshow(cv2.cvtColor(dims_img, cv2.COLOR_BGR2RGB)); ax[5].set_title("dimensions"); ax[5].axis("off")

    supt = f"Center(m): {metrics['center_xyz_m']}  |  L:{metrics['L_m']}  W:{metrics['W_m']}  H:{metrics['H_m']}"
    fig.suptitle(supt, fontsize=10)
    plt.tight_layout()
    p_panel = os.path.join(out_dir, f"{basename}_panel.png")
    fig.savefig(p_panel, dpi=300); plt.close(fig)
    paths["panel"] = p_panel

    return paths, metrics



In [7]:
def run_inference(
    image,
    text_prompt: str = "black box",
    box_threshold: float = 0.2,
    text_threshold: float = 0.15,
    show_window: bool = True,
    auto_save: bool = False,
    depth_image_raw=None,
    intrinsics=None,
    depth_scale: float = None,
    out_dir: str = "paper_figs"
):
    import os, time
    import numpy as np, cv2, torch
    from PIL import Image

    # 0) out_dir를 절대경로로 강제 생성 + 경로 로그
    abs_out_dir = os.path.abspath(out_dir)
    if auto_save:
        os.makedirs(abs_out_dir, exist_ok=True)
        print(f"[auto_save] out_dir = {abs_out_dir}")

    # 1) 이미지 준비
    image_np_bgr = image
    image_np_rgb = cv2.cvtColor(image_np_bgr, cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(image_np_rgb)
    image_np_for_sam = image_np_bgr

    # 2) SAM2
    sam2_predictor.set_image(image_np_for_sam)

    # 3) Grounding DINO
    inputs = processor(images=image_pil, text=text_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = grounding_model(**inputs)
    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        box_threshold=box_threshold, text_threshold=text_threshold,
        target_sizes=[image_pil.size[::-1]]
    )

    def _empty_masks_like(img):
        H, W = img.shape[:2]
        return np.empty((0, H, W), dtype=bool)

    if (not isinstance(results, list)) or (len(results)==0) or ("boxes" not in results[0]):
        annotated_frame = image_np_bgr.copy()
        if show_window:
            try:
                cv2.imshow("Grounded SAM2 Demo", annotated_frame)
                while True:
                    key = cv2.waitKey(1) & 0xFF
                    if key == 27: break
                    if cv2.getWindowProperty("Grounded SAM2 Demo", cv2.WND_PROP_VISIBLE) < 1: break
                cv2.destroyAllWindows()
            except cv2.error: pass
        return annotated_frame, _empty_masks_like(image_np_bgr), []

    input_boxes = results[0]["boxes"].detach().cpu().numpy()
    class_names = [str(x) for x in results[0].get("labels", [])]
    confidences = results[0]["scores"].detach().cpu().numpy().tolist()
    class_ids = np.arange(len(class_names))
    print(f"[det] boxes: {input_boxes.shape[0]}")

    # 4) SAM2 predict
    masks, scores, logits = sam2_predictor.predict(
        point_coords=None, point_labels=None, box=input_boxes, multimask_output=False
    )
    if hasattr(masks, "detach"):
        masks = masks.detach().cpu().numpy()
    if masks.ndim == 4:  # (N,1,H,W) → (N,H,W)
        masks = masks.squeeze(1)
    if (masks is None) or (masks.size == 0) or (masks.shape[0] != input_boxes.shape[0]):
        print("[warn] invalid masks from SAM2 → using empty")
        masks = _empty_masks_like(image_np_bgr)
    else:
        masks = (masks.astype(np.uint8) > 0)

    # 5) (중요) 더 이상 masks를 None으로 덮어쓰지 말 것!
    # !! 당신 기존 코드의: `masks = None` 줄을 삭제해야 함.

    # 6) 시각화
    img_bgr_annotated = image_np_bgr.copy()
    if input_boxes.shape[0] > 0:
        if masks.shape[0] == input_boxes.shape[0] and masks.size > 0:
            detections = sv.Detections(xyxy=input_boxes, mask=masks, class_id=class_ids)
        else:
            detections = sv.Detections(xyxy=input_boxes, class_id=class_ids)
    else:
        detections = sv.Detections.empty()

    annotated_frame = img_bgr_annotated
    if not detections.is_empty():
        # labels_for_display = [f"{cls} {c:.2f}" for cls, c in zip(class_names, confidences)]
        labels_for_display = list(map(str, range(len(class_names))))
        from supervision.draw.color import ColorPalette
        color_palette = ColorPalette.from_hex(CUSTOM_COLORS)
        annotated_frame = sv.BoxAnnotator(color=color_palette).annotate(annotated_frame, detections)
        annotated_frame = sv.LabelAnnotator(color=color_palette).annotate(annotated_frame, detections, labels_for_display)
        # if masks.shape[0] > 0:
        #     annotated_frame = sv.MaskAnnotator(color=color_palette).annotate(annotated_frame, detections) # sam 마스크 영역 색칠 쿄드

    # 7) 자동 저장 조건/로그
    if auto_save:
        cond = {
            "masks>0": masks.shape[0] > 0,
            "depth": depth_image_raw is not None,
            "intrinsics": intrinsics is not None,
            "depth_scale": (depth_scale is not None),
        }
        print(f"[auto_save] conditions = {cond}")
        if all(cond.values()):
            tstamp = time.strftime("%Y%m%d_%H%M%S")
            saved_any = False
            def _match_keyword(name: str, keyword: str) -> bool:
                if name is None: return False
                n = str(name).lower().strip()
                k = keyword.lower().strip()
                aliases = {k, k.replace(" ", ""), k.replace("-", " "), k.replace(" ", "-")}
                return (n in aliases) or any(a in n for a in aliases)

            for i, m in enumerate(masks):
                name = class_names[i] if i < len(class_names) else None
                if not _match_keyword(name, text_prompt):  # 키워드 매칭
                    continue
                if m.sum() < 10:  # 너무 작은 마스크 무시
                    continue
                base = f"{text_prompt.replace(' ','_')}_{tstamp}_{i:02d}"
                try:
                    paths, metrics = save_visuals_for_paper(
                        out_dir=abs_out_dir,
                        basename=base,
                        image_bgr=image_np_bgr,
                        mask=m,
                        depth_image_raw=depth_image_raw,
                        intrinsics=intrinsics,
                        depth_scale=depth_scale
                    )
                    print(f"[auto_save] saved → {paths.get('panel', 'no_panel')} | metrics={metrics}")
                    saved_any = True
                except Exception as e:
                    print(f"[auto_save][error] {e}")
            if not saved_any:
                # 폴더가 안 보였다는 문제 방지: 최소 더미 파일 하나 저장
                dummy_path = os.path.join(abs_out_dir, f"__no_match__{tstamp}.txt")
                with open(dummy_path, "w") as f:
                    f.write("no mask matched the keyword")
                print(f"[auto_save] no match; wrote {dummy_path}")
        else:
            print("[auto_save] skipped (조건 미충족). 폴더는 이미 생성됨:", abs_out_dir)

    if show_window:
        win = "Grounded SAM2 Demo"
        cv2.imshow(win, annotated_frame)
        while True:
            key = cv2.waitKey(1) & 0xFF
            if key == 27: break
            try:
                if cv2.getWindowProperty(win, cv2.WND_PROP_VISIBLE) < 1: break
            except cv2.error:
                break
        cv2.destroyAllWindows()

    return annotated_frame, masks, class_names


In [8]:
# ===== Auto-detect & save for keyword =====
import os, time
import numpy as np

def _normalize_bool_mask(m):
    return (m > 0) if m.dtype != bool else m

def _match_keyword(name: str, keyword: str) -> bool:
    if name is None: 
        return False
    n = str(name).lower().strip()
    k = keyword.lower().strip()
    # 간단한 변형 허용
    aliases = {k, k.replace(" ", ""), k.replace("-", " "), k.replace(" ", "-")}
    return n in aliases or any(a in n for a in aliases)

def auto_save_for_keyword(
    image_bgr,
    depth_image_raw,
    intrinsics,
    depth_scale: float,
    keyword: str = "black box",
    out_dir: str = "paper_figs",
    box_threshold: float = 0.25,   # GroundingDINO 박스 점수 하한
    text_threshold: float = 0.20,  # 텍스트 점수 하한
):
    """
    keyword로 탐지 후, 해당 라벨의 마스크들에 대해 논문용 시각자료를 자동 저장.
    반환: 저장된 (basename -> paths/metrics) 딕셔너리
    """
    os.makedirs(out_dir, exist_ok=True)

    # 1) 탐지 실행 (여기서 run_inference는 (annot_img, masks, class_names) 반환한다고 가정)
    # 이 문장은 제가 추정한 겁니다: 당신의 run_inference 시그니처가 아래와 유사하다.
    result = run_inference(
        image=image_bgr,
        text_prompt=keyword,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
        show_window=False
    )

    # 2) 반환 형태 정규화
    if isinstance(result, tuple) and len(result) >= 3:
        annotated_img, masks, class_names = result[:3]
    else:
        raise RuntimeError(
            "run_inference가 (annotated_img, masks, class_names)를 반환하도록 수정해라. "
            "지금은 한 개만 반환하는 것으로 보인다."
        )

    # 3) 라벨로 필터링
    saved = {}
    tstamp = time.strftime("%Y%m%d_%H%M%S")
    num = 0
    for i, m in enumerate(masks):
        name = class_names[i] if i < len(class_names) else None
        if not _match_keyword(name, keyword):
            continue
        m = _normalize_bool_mask(m)
        if m.sum() < 10:  # 너무 작은 마스크 무시
            continue

        basename = f"{keyword.replace(' ','_')}_{tstamp}_{num:02d}"
        paths, metrics = save_visuals_for_paper(
            out_dir=out_dir,
            basename=basename,
            image_bgr=image_bgr,
            mask=m,
            depth_image_raw=depth_image_raw,
            intrinsics=intrinsics,
            depth_scale=depth_scale
            # center_3d/obb_dims 미지정 → 내부에서 calculate_object_properties로 계산
        )
        saved[basename] = {"paths": paths, "metrics": metrics}
        num += 1

    return saved


In [10]:
# --- 1회 캡처 & 자동 저장 테스트 ---
capture = RealSenseCapture()
assert capture.start(), "RealSense 시작 실패"

color_image, depth_image_raw = capture.capture_aligned_frames()
assert color_image is not None and depth_image_raw is not None, "프레임 캡처 실패"

intrinsics  = capture.get_intrinsics()     # .fx .fy .ppx .ppy
depth_scale = capture.get_depth_scale()    # float (예: 0.001)

print("intrinsics:", intrinsics)
print("depth_scale:", depth_scale)

annot, masks, names = run_inference(
    image=color_image,
    text_prompt="black box",
    box_threshold=0.2,
    text_threshold=0.15,
    show_window=False,
    auto_save=True,                       # ← 자동 저장 켬
    depth_image_raw=depth_image_raw,      # ← 깊이 원본
    intrinsics=intrinsics,                # ← 컬러 intrinsics (정렬 기준이 color니까 이게 맞음)
    depth_scale=depth_scale,              # ← m/단위 스케일
    out_dir="paper_figs"                  # 원하는 폴더
)

capture.stop()
print("결과 라벨:", names)


스트림 설정 시도...
스트림 설정 완료.
파이프라인 시작 중...
파이프라인 시작 완료.
Depth Scale: 0.0010000000474974513
Color Intrinsics: fx=605.5885620117188, fy=605.7113647460938, cx=430.3742980957031, cy=249.6859588623047
초기 안정화 대기 중...
초기 안정화 완료.
intrinsics: [ 848x480  p[430.374 249.686]  f[605.589 605.711]  Inverse Brown Conrady [0 0 0 0 0] ]
depth_scale: 0.0010000000474974513
[auto_save] out_dir = /home/user/문서/neuromeka_ojh/new code/code/workspace/mcp_servers/detect_server/paper_figs
[det] boxes: 1
[auto_save] conditions = {'masks>0': True, 'depth': True, 'intrinsics': True, 'depth_scale': True}
[auto_save] saved → /home/user/문서/neuromeka_ojh/new code/code/workspace/mcp_servers/detect_server/paper_figs/black_box_20250817_143853_00_panel.png | metrics={'center_xyz_m': [-0.07233632355928421, -0.020373350009322166, 0.5204614400863647], 'L_m': 0.1234903484582901, 'W_m': 0.06393840163946152, 'H_m': 0.02399998903274536, 'used_calculate_object_properties': True}
파이프라인 중지...
파이프라인 중지 완료.
결과 라벨: ['black box']


In [ ]:
# === LAST CELL (histogram removed): capture → detect → save paper-figs ======
import os, time, json, cv2, numpy as np
import matplotlib.pyplot as plt

# ---------- helpers ----------
def _ensure_bool_mask(mask):
    m = np.asarray(mask)
    return m.astype(bool) if m.dtype != bool else m

def _depth_to_m(depth_image_raw, depth_scale: float):
    return depth_image_raw.astype(np.float32) * float(depth_scale)

def _masked_points_xyz_from_intrinsics(mask_bool, depth_m, intrinsics,
                                       z_min=0.01, z_max=0.8):
    mask_bool = _ensure_bool_mask(mask_bool)
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return np.empty((0,3), np.float32), (np.array([], dtype=np.int32), np.array([], dtype=np.int32))
    z = depth_m[ys, xs].astype(np.float32)
    valid = (z > z_min) & (z < z_max)
    if valid.sum() < 4:
        return np.empty((0,3), np.float32), (np.array([], dtype=np.int32), np.array([], dtype=np.int32))
    xs = xs[valid].astype(np.float32); ys = ys[valid].astype(np.float32); z = z[valid]
    fx, fy = intrinsics.fx, intrinsics.fy
    cx, cy = intrinsics.ppx, intrinsics.ppy
    X = (xs - cx) * z / fx
    Y = (ys - cy) * z / fy
    Z = z
    pc = np.stack([X,Y,Z], axis=1).astype(np.float32)
    return pc, (xs.astype(np.int32), ys.astype(np.int32))

def _oriented_bbox(mask_bool):
    m = (_ensure_bool_mask(mask_bool).astype(np.uint8))*255
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None
    cnt = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)
    box  = cv2.boxPoints(rect).astype(np.int32)
    return rect, box

def _depth_grayscale_img(depth_m: np.ndarray, mask_bool: np.ndarray):
    # 마스크 구간 2~98% 범위로 스트레치, 흑백 출력
    m = _ensure_bool_mask(mask_bool)
    vals = depth_m[m]
    if vals.size == 0:
        return None
    lo, hi = np.percentile(vals, [2, 98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(vals.min()), float(vals.max())
        if hi <= lo: return None
    norm = np.clip((depth_m - lo) / max(1e-6, (hi - lo)), 0, 1)
    gray8 = (norm * 255).astype(np.uint8)
    return gray8

def _save_pointcloud_fig(pc_xyz, out_path, max_points=15000):
    if pc_xyz.shape[0] == 0:
        return False
    sel = np.random.choice(pc_xyz.shape[0], size=min(max_points, pc_xyz.shape[0]), replace=False)
    P = pc_xyz[sel].copy()

    # --- 시각화용 Z축 반전 (계산/메트릭에는 영향 없음) ---
    P[:, 2] = -P[:, 2]

    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    fig = plt.figure(figsize=(4,4))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(P[:,0], P[:,1], P[:,2], s=1)

    # 보기 각도(선택): 필요 없으면 지워도 됨
    ax.view_init(elev=18, azim=35)

    # --- 축/격자/프레임 완전 숨김 ---
    ax.grid(False)
    try:
        ax.set_axis_off()
    except Exception:
        pass
    # pane/라인 숨김(버전 호환)
    for attr in ("xaxis", "yaxis", "zaxis"):
        axis = getattr(ax, attr, None)
        if axis is not None:
            try:
                axis.set_ticks([])
            except Exception:
                pass
            try:
                axis.pane.set_visible(False)
            except Exception:
                pass
    for attr in ("w_xaxis", "w_yaxis", "w_zaxis"):
        waxis = getattr(ax, attr, None)
        if waxis is not None:
            try:
                waxis.line.set_visible(False)
            except Exception:
                pass

    plt.tight_layout(pad=0.0)
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    return True

def save_visuals_for_paper(
    out_dir: str,
    basename: str,
    image_bgr: np.ndarray,
    mask: np.ndarray,
    depth_image_raw: np.ndarray,
    intrinsics,
    depth_scale: float,
    center_3d=None,      # (x,y,z) m
    obb_dims=None,       # [L, W, H] m
    image_with_boxes: np.ndarray = None,   # run_inference의 annotated_frame
    boxes_xyxy: np.ndarray = None,         # annotated_frame이 없으면 박스를 여기로 전달
    box_labels: list = None
):
    """
    - calculate_object_properties(...) (네가 위에 정의한 함수)를 값 미지정 시 내부 호출.
    - overlay 제거, 깊이는 흑백, 포인트클라우드 grid 제거, 히스토그램 제거.
    - 패널 좌상단은 RGB + DINO boxes.
    """
    os.makedirs(out_dir, exist_ok=True)
    paths = {}

    # 0) 이진 마스크 저장
    mask_bool = _ensure_bool_mask(mask)
    mask_u8 = (mask_bool.astype(np.uint8))*255
    p_mask = os.path.join(out_dir, f"{basename}_mask.png")
    cv2.imwrite(p_mask, mask_u8); paths["mask"] = p_mask

    # 1) RGB + DINO 박스 이미지 저장
    if image_with_boxes is not None:
        rgb_boxes_img = image_with_boxes.copy()
    else:
        rgb_boxes_img = image_bgr.copy()
        if boxes_xyxy is not None and len(np.atleast_2d(boxes_xyxy)) > 0:
            boxes_xyxy = np.asarray(boxes_xyxy, dtype=np.float32)
            for i, (x1, y1, x2, y2) in enumerate(boxes_xyxy.astype(int)):
                cv2.rectangle(rgb_boxes_img, (x1, y1), (x2, y2), (0, 255, 255), 2)
                if box_labels is not None and i < len(box_labels):
                    cv2.putText(rgb_boxes_img, str(box_labels[i]), (x1, max(0,y1-5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1, cv2.LINE_AA)
    p_rgb_boxes = os.path.join(out_dir, f"{basename}_rgb_boxes.png")
    cv2.imwrite(p_rgb_boxes, rgb_boxes_img); paths["rgb_boxes"] = p_rgb_boxes

    # 2) 깊이(m) 및 포인트클라우드
    depth_m = _depth_to_m(depth_image_raw, depth_scale)
    pc_xyz, _ = _masked_points_xyz_from_intrinsics(mask_bool, depth_m, intrinsics)
    has_pc = pc_xyz.shape[0] > 0

    # 3) 깊이 흑백 맵 (히스토그램 제거)
    gray = _depth_grayscale_img(depth_m, mask_bool)
    if gray is not None:
        p_dgray = os.path.join(out_dir, f"{basename}_depth_gray.png")
        cv2.imwrite(p_dgray, gray); paths["depth_gray"] = p_dgray

    # 4) 3D 수치 없으면 계산 (네 함수 사용)
    used_calc = False
    if (center_3d is None) or (obb_dims is None):
        try:
            center_3d2, obb_dims2, tilt_deg, rot_deg = calculate_object_properties(
                mask_bool, depth_image_raw, intrinsics, depth_scale
            )
            if center_3d is None: center_3d = center_3d2
            if obb_dims is None:  obb_dims  = obb_dims2
            used_calc = True
        except Exception as e:
            print(f"[warn] calculate_object_properties failed: {e}")

    # 5) 포인트클라우드 그림 (grid 끔)
    if has_pc:
        p_pc = os.path.join(out_dir, f"{basename}_pointcloud.png")
        if _save_pointcloud_fig(pc_xyz, p_pc):
            paths["pointcloud"] = p_pc

    # 6) 2D OBB + L/W 텍스트
    p_dims2d = os.path.join(out_dir, f"{basename}_dims2d.png")
    dims_img = image_bgr.copy()
    rect, box = _oriented_bbox(mask_bool)
    if rect is not None and box is not None:
        cv2.polylines(dims_img, [box], True, (255,0,255), 2)
        p = box
        mid01 = tuple(np.mean([p[0],p[1]], axis=0).astype(int))
        mid12 = tuple(np.mean([p[1],p[2]], axis=0).astype(int))
        L_txt = f"L={obb_dims[0]:.3f} m" if (obb_dims is not None) else "L=N/A"
        W_txt = f"W={obb_dims[1]:.3f} m" if (obb_dims is not None) else "W=N/A"
        cv2.putText(dims_img, L_txt, mid01, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,255), 2, cv2.LINE_AA)
        cv2.putText(dims_img, W_txt, mid12, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,255), 2, cv2.LINE_AA)
    cv2.imwrite(p_dims2d, dims_img); paths["dims2d"] = p_dims2d

    # 7) metrics JSON
    metrics = {
        "center_xyz_m": None if center_3d is None else [float(center_3d[0]), float(center_3d[1]), float(center_3d[2])],
        "L_m": None if (obb_dims is None) else float(obb_dims[0]),
        "W_m": None if (obb_dims is None) else float(obb_dims[1]),
        "H_m": None if (obb_dims is None) else float(obb_dims[2]),
        "used_calculate_object_properties": bool(used_calc),
    }
    p_json = os.path.join(out_dir, f"{basename}_metrics.json")
    with open(p_json, "w") as f:
        json.dump(metrics, f, indent=2)
    paths["metrics_json"] = p_json

    # 8) 패널 합성(6칸) — 히스토그램 자리 제거(공란)
    import matplotlib.image as mpimg
    fig, axes = plt.subplots(2,3, figsize=(10,6))
    ax = axes.ravel()

    ax[0].imshow(cv2.cvtColor(rgb_boxes_img, cv2.COLOR_BGR2RGB)); ax[0].set_title("RGB + DINO boxes"); ax[0].axis("off")
    ax[1].imshow(mask_bool, cmap="gray"); ax[1].set_title("Binary mask"); ax[1].axis("off")

    if "depth_gray" in paths:
        dimg = mpimg.imread(paths["depth_gray"])
        ax[2].imshow(dimg, cmap="gray"); ax[2].set_title("Depth (grayscale)"); ax[2].axis("off")
    else:
        ax[2].axis("off")

    # (1,0): Masked point cloud
    if "pointcloud" in paths:
        pcimg = mpimg.imread(paths["pointcloud"])
        ax[3].imshow(pcimg); ax[3].set_title("Masked point cloud"); ax[3].axis("off")
    else:
        ax[3].axis("off")

    # (1,1): 2D dims
    ax[4].imshow(cv2.cvtColor(dims_img, cv2.COLOR_BGR2RGB)); ax[4].set_title("dimensions"); ax[4].axis("off")

    # (1,2): 빈 칸
    ax[5].axis("off")

    supt = f"Center(m): {metrics['center_xyz_m']}  |  L:{metrics['L_m']}  W:{metrics['W_m']}  H:{metrics['H_m']}"
    fig.suptitle(supt, fontsize=10)
    plt.tight_layout()
    p_panel = os.path.join(out_dir, f"{basename}_panel.png")
    fig.savefig(p_panel, dpi=300); plt.close(fig)
    paths["panel"] = p_panel

    return paths, metrics
# ---------------------------------------------------------------------------

# ---------- RUN ONCE: capture → inference → save ----------
TEXT_PROMPT = "can"     # 탐지 키워드
OUT_DIR = "paper_figs"         # 출력 폴더
MIN_MASK_PIXELS = 10           # 너무 작은 마스크 필터

# 1) 카메라 시작 & 프레임 획득
cap = RealSenseCapture()
assert cap.start(), "RealSense 시작 실패"
color_image, depth_image_raw = cap.capture_aligned_frames()
assert color_image is not None and depth_image_raw is not None, "프레임 캡처 실패"
intrinsics  = cap.get_intrinsics()
depth_scale = cap.get_depth_scale()
print(f"[info] intrinsics: fx={intrinsics.fx}, fy={intrinsics.fy}, cx={intrinsics.ppx}, cy={intrinsics.ppy}")
print(f"[info] depth_scale: {depth_scale}")

# 2) 추론 (창 표시 없이, 자동저장 off — 여기서 수동 저장)
annotated_frame, masks, class_names = run_inference(
    image=color_image,
    text_prompt=TEXT_PROMPT,
    box_threshold=0.1,
    text_threshold=0.1,
    show_window=False,
    auto_save=False
)
cap.stop()

# 3) 저장 루프 (라벨 매칭 없이 모든 마스크 저장)
os.makedirs(OUT_DIR, exist_ok=True)
tstamp = time.strftime("%Y%m%d_%H%M%S")
saved = []
for i, m in enumerate(masks):
    if m is None: 
        continue
    m = _ensure_bool_mask(m)
    if m.sum() < MIN_MASK_PIXELS:
        continue
    base = f"{TEXT_PROMPT.replace(' ','_')}_{tstamp}_{i:02d}"
    paths, metrics = save_visuals_for_paper(
        out_dir=OUT_DIR,
        basename=base,
        image_bgr=color_image,
        mask=m,
        depth_image_raw=depth_image_raw,
        intrinsics=intrinsics,
        depth_scale=depth_scale,
        image_with_boxes=annotated_frame  # ★ 박스 포함 RGB 사용
    )
    saved.append((base, paths.get("panel"), metrics))

# 4) 결과 요약
print("\n=== SAVED ===")
if not saved:
    print("저장된 패널 없음 (마스크가 없거나 너무 작음).")
else:
    for base, panel_path, metrics in saved:
        print(f"{base} → {panel_path}")
        print("metrics:", metrics)


스트림 설정 시도...
스트림 설정 완료.
파이프라인 시작 중...
파이프라인 시작 완료.
Depth Scale: 0.0010000000474974513
Color Intrinsics: fx=605.5885620117188, fy=605.7113647460938, cx=430.3742980957031, cy=249.6859588623047
초기 안정화 대기 중...
초기 안정화 완료.
[info] intrinsics: fx=605.5885620117188, fy=605.7113647460938, cx=430.3742980957031, cy=249.6859588623047
[info] depth_scale: 0.0010000000474974513
[det] boxes: 6
파이프라인 중지...
파이프라인 중지 완료.

=== SAVED ===
can_20250817_143932_00 → paper_figs/can_20250817_143932_00_panel.png
metrics: {'center_xyz_m': [-0.0064309826120734215, 0.13157548010349274, 0.42838647961616516], 'L_m': 0.09920105338096619, 'W_m': 0.05968703702092171, 'H_m': 0.13300001621246338, 'used_calculate_object_properties': True}
can_20250817_143932_01 → paper_figs/can_20250817_143932_01_panel.png
metrics: {'center_xyz_m': [0.15553699433803558, -0.04076259955763817, 0.5265972018241882], 'L_m': 0.10266159474849701, 'W_m': 0.059647902846336365, 'H_m': 0.03299999237060547, 'used_calculate_object_properties': True}
can_

: 